In [ ]:
import pandas as pd
import math
from datetime import datetime, timedelta
from difflib import SequenceMatcher
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from lifelines import CoxPHFitter
import warnings
from preproces_prod3 import *
from patsy import dmatrices
warnings.filterwarnings("ignore")
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Configurar las credenciales
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/spreadsheets",
         "https://www.googleapis.com/auth/drive.file", "https://www.googleapis.com/auth/drive"]

creds = ServiceAccountCredentials.from_json_keyfile_name(path_actual.parent.parent/"credencials.json", scope)
client = gspread.authorize(creds)

# Abrir la hoja de cálculo
spreadsheet = client.open("Modelos_covariables_nirse")  # Reemplaza con el nombre de tu hoja de cálculo
sheet = spreadsheet.sheet1 


from gspread_dataframe import set_with_dataframe

In [ ]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Configurar las credenciales
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/spreadsheets",
         "https://www.googleapis.com/auth/drive.file", "https://www.googleapis.com/auth/drive"]

creds = ServiceAccountCredentials.from_json_keyfile_name(path_actual.parent.parent/"credencials.json", scope)
client = gspread.authorize(creds)

# Abrir la hoja de cálculo
spreadsheet = client.open("Modelos_covariables_nirse")  # Reemplaza con el nombre de tu hoja de cálculo
sheet = spreadsheet.sheet1 

In [8]:
def upload_results_to_sheet(spreadsheet, df, sheet_name):
    """
    Sube un DataFrame a una hoja específica en un archivo de Google Sheets.
    
    Args:
        spreadsheet: Objeto de la hoja de cálculo (spreadsheet) abierta con gspread.
        df: DataFrame a subir.
        sheet_name: Nombre de la hoja dentro del archivo de Google Sheets.
    """
    try:
        # Abrir o crear la hoja especificada
        try:
            worksheet = spreadsheet.worksheet(sheet_name)
        except gspread.exceptions.WorksheetNotFound:
            worksheet = spreadsheet.add_worksheet(title=sheet_name, rows=100, cols=20)

        # Limpiar la hoja antes de cargar nuevos datos
        worksheet.clear()

        # Subir el DataFrame a la hoja
        set_with_dataframe(worksheet, df)

        print(f"Datos subidos exitosamente a la hoja '{sheet_name}'")
    except Exception as e:
        print(f"Error al subir los datos: {e}")

In [2]:
df_vrs = pd.read_csv(path_data/'df_vrs_s34_1811_all_meses.csv')

In [ ]:
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_vrs[['start', 'inmunizado', 'stop', 'RUN', 'event_vrs', 'sexo', 'muy_prematuro', 'region','group_age']], 
          id_col="RUN", event_col="event_vrs", start_col="start", stop_col="stop", strata=['region','group_age'])
print("STRATIFICATION")
print("N_bebes_total:", df_vrs.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", df_vrs.drop_duplicates(subset='RUN', keep='last').inmunizado.sum())
print("Cobertura =", round(df_vrs.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                           /df_vrs.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS (inm, no inm):", df_vrs.drop_duplicates(subset='RUN', keep='last').event_vrs.sum() ,f'({df_vrs.drop_duplicates(subset="RUN", keep="last").query("inmunizado==1").event_vrs.sum()}, {df_vrs.drop_duplicates(subset="RUN", keep="last").query("inmunizado==0").event_vrs.sum()})')
print("Ratio: VRS Inm  / VRS no Inm:", df_vrs.drop_duplicates(subset='RUN', keep='last').query('inmunizado==1').event_vrs.sum() / 
        df_vrs.drop_duplicates(subset='RUN', keep='last').query('inmunizado==0').event_vrs.sum())
print("Rato_VRS/N:", round(df_vrs.drop_duplicates(subset='RUN', keep='last').event_vrs.sum()
        / df_vrs.drop_duplicates(subset='RUN', keep='last').shape[0],5)*100)
display(printSummary(ctv_0))
print('\n')


for i in df_vrs.region.unique():
    ctv_0 = CoxTimeVaryingFitter()
    if pd.isnull(i):
        continue
    else:
        print("SUBSET_REGION:", i)
        d_reg=df_vrs.query('region==' + '"'+ i +'"')
        ctv_0.fit(d_reg[['start', 'inmunizado', 'stop', 'RUN', 'event_vrs', 'group_age', 'sexo']], #,'muy_prematuro'
                id_col="RUN", event_col="event_vrs", start_col="start", stop_col="stop",strata=['group_age'])
        print("N_bebes_total:", d_reg.drop_duplicates(subset='RUN', keep='last').shape[0])
        print("N_bebes_inmune:", d_reg.drop_duplicates(subset='RUN', keep='last').inmunizado.sum())
        print("Cobertura =",
              round(d_reg.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                    /d_reg.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
        print("N_bebes_VRS (inm, no inm):", d_reg.drop_duplicates(subset='RUN', keep='last').event_vrs.sum() ,f'({d_reg.drop_duplicates(subset="RUN", keep="last").query("inmunizado==1").event_vrs.sum()}, {d_reg.drop_duplicates(subset="RUN", keep="last").query("inmunizado==0").event_vrs.sum()})')
        print("Ratio: VRS Inm  / VRS no Inm:", round(d_reg.drop_duplicates(subset='RUN', keep='last').query('inmunizado==1').event_vrs.sum() / 
              d_reg.drop_duplicates(subset='RUN', keep='last').query('inmunizado==0').event_vrs.sum(),1))
        print("Rato_VRS/N:", round(d_reg.drop_duplicates(subset='RUN', keep='last').event_vrs.sum()
              / d_reg.drop_duplicates(subset='RUN', keep='last').shape[0],5)*100)
        display(printSummary(ctv_0))
        print('\n')


STRATIFICATION
N_bebes_total: 141395
N_bebes_inmune: 129873.0
Cobertura = 91.85 %
N_bebes_VRS (inm, no inm): 867 (583, 284)
Ratio: VRS Inm  / VRS no Inm: 2.0528169014084505
Rato_VRS/N: 0.613


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.357,0.743,0.081,-1.517,-1.198,0.698,0.781,0.0
sexo,0.523,-0.686,0.071,0.384,0.661,-0.937,-0.468,0.0
muy_prematuro,1.128,-2.090,0.165,0.805,1.451,-3.269,-1.236,0.0




SUBSET_REGION: METROPOLITANA
N_bebes_total: 55824
N_bebes_inmune: 52554.0
Cobertura = 94.14 %
N_bebes_VRS (inm, no inm): 429 (291, 138)
Ratio: VRS Inm  / VRS no Inm: 2.1
Rato_VRS/N: 0.768


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.545,0.787,0.115,-1.772,-1.319,0.733,0.830,0.0
sexo,0.496,-0.642,0.100,0.300,0.692,-0.997,-0.349,0.0




SUBSET_REGION: ATACAMA
N_bebes_total: 2696
N_bebes_inmune: 2411.0
Cobertura = 89.43 %
N_bebes_VRS (inm, no inm): 9 (7, 2)
Ratio: VRS Inm  / VRS no Inm: 3.5
Rato_VRS/N: 0.334


...

,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.316,0.732,0.393,-2.085,-0.546,0.421,0.876,0.001
sexo,0.063,-0.065,0.329,-0.582,0.708,-1.030,0.441,0.849




SUBSET_REGION: ARICA Y PARINACOTA
N_bebes_total: 2207
N_bebes_inmune: 1911.0
Cobertura = 86.59 %
N_bebes_VRS (inm, no inm): 7 (2, 5)
Ratio: VRS Inm  / VRS no Inm: 0.4
Rato_VRS/N: 0.317


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-1.447,0.765,0.991,-3.389,0.495,-0.640,0.966,0.144
sexo,0.902,-1.465,0.838,-0.740,2.544,-11.731,0.523,0.282




SUBSET_REGION: MAGALLANES Y ANTARTICA
N_bebes_total: 1163
N_bebes_inmune: 1093.0
Cobertura = 93.98 %
N_bebes_VRS (inm, no inm): 6 (4, 2)
Ratio: VRS Inm  / VRS no Inm: 2.0
Rato_VRS/N: 0.516


,coef,effectiveness,se(coef),coef lower 95%,coef upper 95%,eff_lower_95,eff_upper_95,p
covariate,,,,,,,,
inmunizado,-0.812,0.556,0.918,-2.612,0.989,-1.687,0.927,0.377
sexo,-0.063,0.061,0.821,-1.672,1.547,-3.696,0.812,0.939


In [ ]:
######################################################################### TO SHEETS ##################################################################################

Celda='B'
n_celda=2

ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(d[['start', 'inmunizado', 'stop', 'RUN', 'event', 'si_3_meses', 'SEXO', 'muy_prematuro', 'NOMBRE_REGION']], 
          id_col="RUN", event_col="event", start_col="start", stop_col="stop", strata=['NOMBRE_REGION'])
print("STRATIFICATION")
print("N_bebes_total:", d.drop_duplicates(subset='RUN', keep='last').shape[0])
print("N_bebes_inmune:", d.drop_duplicates(subset='RUN', keep='last').inmunizado.sum())
print("Cobertura =", round(d.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                           /d.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
print("N_bebes_VRS (inm, no inm):", d.drop_duplicates(subset='RUN', keep='last').event.sum() ,f'({d.drop_duplicates(subset="RUN", keep="last").query("inmunizado==1").event.sum()}, {d.drop_duplicates(subset="RUN", keep="last").query("inmunizado==0").event.sum()})')
print("Ratio: VRS Inm  / VRS no Inm:", d.drop_duplicates(subset='RUN', keep='last').query('inmunizado==1').event.sum() / 
        d.drop_duplicates(subset='RUN', keep='last').query('inmunizado==0').event.sum())
print("Rato_VRS/N:", round(d.drop_duplicates(subset='RUN', keep='last').event.sum()
        / d.drop_duplicates(subset='RUN', keep='last').shape[0],5)*100)
display(printSummary(ctv_0))
print('\n')
worksheet = spreadsheet.worksheet("Regions_31")
model = printSummary(ctv_0)
df_values = model.reset_index().values.tolist()
df_headers = ['Covariates'] + model.columns.tolist()
worksheet.update(Celda+str(n_celda), [df_headers] + df_values)
n_celda = n_celda + len(df_values) + 2 


for i in d.NOMBRE_REGION.unique():
    ctv_0 = CoxTimeVaryingFitter()
    if pd.isnull(i):
        continue
    else:
        print("SUBSET_REGION:", i)
        d_reg=d.query('NOMBRE_REGION==' + '"'+ i +'"')
        ctv_0.fit(d_reg[['start', 'inmunizado', 'stop', 'RUN', 'event', 'si_3_meses', 'SEXO']], #,'muy_prematuro'
                id_col="RUN", event_col="event", start_col="start", stop_col="stop")
        print("N_bebes_total:", d_reg.drop_duplicates(subset='RUN', keep='last').shape[0])
        print("N_bebes_inmune:", d_reg.drop_duplicates(subset='RUN', keep='last').inmunizado.sum())
        print("Cobertura =",
              round(d_reg.drop_duplicates(subset='RUN', keep='last').inmunizado.sum() * 100 
                    /d_reg.drop_duplicates(subset='RUN', keep='last').shape[0],2),"%")
        print("N_bebes_VRS (inm, no inm):", d_reg.drop_duplicates(subset='RUN', keep='last').event.sum() ,f'({d_reg.drop_duplicates(subset="RUN", keep="last").query("inmunizado==1").event.sum()}, {d_reg.drop_duplicates(subset="RUN", keep="last").query("inmunizado==0").event.sum()})')
        print("Ratio: VRS Inm  / VRS no Inm:", round(d_reg.drop_duplicates(subset='RUN', keep='last').query('inmunizado==1').event.sum() / 
              d_reg.drop_duplicates(subset='RUN', keep='last').query('inmunizado==0').event.sum(),1))
        print("Rato_VRS/N:", round(d_reg.drop_duplicates(subset='RUN', keep='last').event.sum()
              / d_reg.drop_duplicates(subset='RUN', keep='last').shape[0],5)*100)
        display(printSummary(ctv_0))
        print('\n')
        worksheet = spreadsheet.worksheet("Regions_31")
        model = printSummary(ctv_0).replace([np.inf, -np.inf], np.nan).fillna(0)

        df_values = model.reset_index().values.tolist()
        df_headers = ['Covariates'] + model.columns.tolist()

        worksheet.update(Celda+str(n_celda), [df_headers] + df_values)
        n_celda = n_celda + len(df_values) + 2 

In [9]:
# DataFrame para resultados
results = []

# STRATIFICATION
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_vrs[['start', 'inmunizado', 'stop', 'RUN', 'event_vrs', 'sexo', 'muy_prematuro', 'region', 'group_age']], 
          id_col="RUN", event_col="event_vrs", start_col="start", stop_col="stop", strata=['region', 'group_age'])

strat_total = df_vrs.drop_duplicates(subset='RUN', keep='last')
results.append({
    'SUBSET_REGION': 'STRATIFICATION',
    'N_bebes_total': strat_total.shape[0],
    'N_bebes_inmune': strat_total.inmunizado.sum(),
    'Cobertura (%)': round(strat_total.inmunizado.sum() * 100 / strat_total.shape[0], 2),
    'N_bebes_VRS (inm, no inm)': f"{strat_total.event_vrs.sum()} ({strat_total.query('inmunizado==1').event_vrs.sum()}, {strat_total.query('inmunizado==0').event_vrs.sum()})",
    'Ratio VRS Inm / VRS no Inm': round(
        strat_total.query('inmunizado==1').event_vrs.sum() / strat_total.query('inmunizado==0').event_vrs.sum(), 2),
    'Rato_VRS/N': round(strat_total.event_vrs.sum() / strat_total.shape[0], 5) * 100
})

# Por región
for region in df_vrs.region.unique():
    if pd.isnull(region):
        continue
    d_reg = df_vrs.query(f'region == "{region}"')
    ctv_0.fit(d_reg[['start', 'inmunizado', 'stop', 'RUN', 'event_vrs', 'group_age', 'sexo']], 
              id_col="RUN", event_col="event_vrs", start_col="start", stop_col="stop", strata=['group_age'])
    
    region_total = d_reg.drop_duplicates(subset='RUN', keep='last')
    results.append({
        'SUBSET_REGION': region,
        'N_bebes_total': region_total.shape[0],
        'N_bebes_inmune': region_total.inmunizado.sum(),
        'Cobertura (%)': round(region_total.inmunizado.sum() * 100 / region_total.shape[0], 2),
        'N_bebes_VRS (inm, no inm)': f"{region_total.event_vrs.sum()} ({region_total.query('inmunizado==1').event_vrs.sum()}, {region_total.query('inmunizado==0').event_vrs.sum()})",
        'Ratio VRS Inm / VRS no Inm': round(
            region_total.query('inmunizado==1').event_vrs.sum() / region_total.query('inmunizado==0').event_vrs.sum(), 2),
        'Rato_VRS/N': round(region_total.event_vrs.sum() / region_total.shape[0], 5) * 100
    })

# Convertir resultados a DataFrame
results_df = pd.DataFrame(results)


In [14]:
# DataFrame para resultados
results = []

# STRATIFICATION
ctv_0 = CoxTimeVaryingFitter()
ctv_0.fit(df_vrs[['start', 'inmunizado', 'stop', 'RUN', 'event_vrs', 'sexo', 'muy_prematuro', 'region', 'group_age']], 
          id_col="RUN", event_col="event_vrs", start_col="start", stop_col="stop", strata=['region', 'group_age'])

# Extraer coeficiente y valor p para 'inmunizado'
try:
    coef = ctv_0.summary.loc['inmunizado', 'coef']
    p_value = ctv_0.summary.loc['inmunizado', 'p']  # Asegúrate de que la columna se llame 'p' o 'p-value'
    effectiveness = (1 - np.exp(coef)) * 100
except KeyError:
    # Manejar el caso donde 'inmunizado' no está en el resumen
    coef = np.nan
    p_value = np.nan
    effectiveness = np.nan

strat_total = df_vrs.drop_duplicates(subset='RUN', keep='last')
results.append({
    'SUBSET_REGION': 'STRATIFICATION',
    'N_bebes_total': strat_total.shape[0],
    'N_bebes_inmune': strat_total.inmunizado.sum(),
    'Cobertura (%)': round(strat_total.inmunizado.sum() * 100 / strat_total.shape[0], 2),
    'N_bebes_VRS (inm, no inm)': f"{strat_total.event_vrs.sum()} ({strat_total.query('inmunizado==1').event_vrs.sum()}, {strat_total.query('inmunizado==0').event_vrs.sum()})",
    'Ratio VRS Inm / VRS no Inm': round(
        strat_total.query('inmunizado==1').event_vrs.sum() / strat_total.query('inmunizado==0').event_vrs.sum(), 2),
    'Rato_VRS/N': round(strat_total.event_vrs.sum() / strat_total.shape[0], 5) * 100,
    'effectiveness': effectiveness,
    'p_value': p_value
})

# Por región
for region in df_vrs.region.unique():
    if pd.isnull(region):
        continue
    d_reg = df_vrs.query(f'region == "{region}"')
    ctv_0.fit(d_reg[['start', 'inmunizado', 'stop', 'RUN', 'event_vrs', 'group_age', 'sexo']], 
              id_col="RUN", event_col="event_vrs", start_col="start", stop_col="stop", strata=['group_age'])
    
    # Extraer coeficiente y valor p para 'inmunizado'
    try:
        coef = ctv_0.summary.loc['inmunizado', 'coef']
        p_value = ctv_0.summary.loc['inmunizado', 'p']  # Verifica el nombre exacto de la columna
        effectiveness = (1 - np.exp(coef)) * 100
    except KeyError:
        coef = np.nan
        p_value = np.nan
        effectiveness = np.nan

    region_total = d_reg.drop_duplicates(subset='RUN', keep='last')
    results.append({
        'SUBSET_REGION': region,
        'N_bebes_total': region_total.shape[0],
        'N_bebes_inmune': region_total.inmunizado.sum(),
        'Cobertura (%)': round(region_total.inmunizado.sum() * 100 / region_total.shape[0], 2),
        'N_bebes_VRS (inm, no inm)': f"{region_total.event_vrs.sum()} ({region_total.query('inmunizado==1').event_vrs.sum()}, {region_total.query('inmunizado==0').event_vrs.sum()})",
        'Ratio VRS Inm / VRS no Inm': round(
            region_total.query('inmunizado==1').event_vrs.sum() / region_total.query('inmunizado==0').event_vrs.sum(), 2),
        'Rato_VRS/N': round(region_total.event_vrs.sum() / region_total.shape[0], 5) * 100,
        'effectiveness': effectiveness,
        'p_value': p_value
    })

# Convertir resultados a DataFrame
results_df = pd.DataFrame(results)

# Reemplazar valores problemáticos antes de subir
results_df.replace([float('inf'), float('-inf')], np.nan, inplace=True)
results_df.fillna("NA", inplace=True)


In [15]:
results_df.replace([float('inf'), float('-inf')], None, inplace=True)  # O usa un valor como 0 o 'NA'
results_df.fillna("NA", inplace=True)

In [16]:
sheet_name = "regiones_iterable"

# Subir los resultados al archivo en Google Sheets
upload_results_to_sheet(spreadsheet, results_df, sheet_name)

Datos subidos exitosamente a la hoja 'regiones_iterable'
